[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/43.%20The%20Security%20Inspection%20Optimization%20Problem/P43-Tier-3_executed.ipynb)

---

# 43. The Security Inspection Optimization Problem

## Tier 3: Moth-Flame Optimization

### Goal and workflow

This notebook implements Moth-Flame Optimization (MFO), a population-based metaheuristic inspired by moth navigation around artificial lights. It searches for a high-fitness security-screening configuration under a deployment-budget constraint.

The workflow is: define the screening instance; encode each candidate as a binary deployment vector; evaluate its fitness; maintain and reduce a ranked set of guiding flames; move the moths by a logarithmic-spiral rule; and inspect the best feasible configuration and the search diagnostics.

### Inputs and outputs

The instance supplies locations, passenger categories, screening technologies, passenger demands, threat probabilities, detection and false-alarm rates, deployment costs, and a total budget. The algorithm returns its best recorded configuration, its fitness and performance measures, and histories describing the search.

### Model assumptions and scope

A location can receive at most one technology, or none. In this teaching instance, deploying a technology at any location makes that technology available to every passenger category in the aggregate performance calculation. Locations therefore affect deployment cost and the one-technology-per-location restriction, but do not represent separate passenger flows or local capacity. The model does not include routing, throughput limits, interactions between technologies, or uncertainty in the input data. These assumptions define the scope of the example; results should not be interpreted as a detailed operational airport plan.

The budget is a hard feasibility constraint. Detection benefit minus a penalty for false alarms is the fitness objective; deployment cost is not independently subtracted from fitness. A high fitness is desirable, but MFO is a stochastic heuristic and does not certify global optimality.

## Model and notation

Let \(L\) be the set of locations, \(P\) the set of passenger categories, and \(T\) the set of technologies. A binary decision \(x_{lt}\) is 1 when technology \(t\) is installed at location \(l\), and 0 otherwise. The deployment decisions obey

$$
\sum_{t\in T} x_{lt} \leq 1 \quad (l\in L),
\qquad
\sum_{l\in L}\sum_{t\in T} c_{lt}x_{lt} \leq B.
$$

Here, \(c_{lt}\) is deployment cost and \(B\) is the total budget. The first constraint is enforced by the position-repair step; the second is enforced when fitness is evaluated.

For category \(p\), let \(d_p\) be demand, \(q_p\) its threat probability, and let \(a_{pt}\) and \(f_{pt}\) denote detection and false-alarm rates for a compatible technology. Compatibility is represented by the model's compatibility table. Under the aggregate-availability assumption, define the selected technology as the compatible technology with the greatest detection rate among those deployed at one or more locations. If none is available, both rates are zero. Denote the resulting rates by \(\hat a_p(x)\) and \(\hat f_p(x)\). Then

$$
D(x)=\sum_{p\in P}q_p d_p\hat a_p(x),
\qquad
A(x)=\sum_{p\in P}d_p\hat f_p(x),
\qquad
F(x)=D(x)-0.1A(x).
$$

The reported detection rate is \(D(x)/\sum_{p\in P}q_pd_p\), when the denominator is positive. The reported false-alarm rate is \(A(x)/\sum_{p\in P}d_p\), when total demand is positive. The coefficient 0.1 is the false-alarm penalty used by this instance's fitness definition; it is not a monetary cost. In the code, these quantities are calculated by `calculate_fitness`.

## MFO search logic

Intuitively, each moth is one candidate configuration and each flame is a promising configuration that guides moth movement. The algorithm ranks candidates by fitness, moves each moth relative to a selected flame, and gradually reduces the number of flames to emphasize the leading candidates. This gives the search a population of evolving candidates while retaining a best-so-far solution for reporting.

A moth position is a binary vector of length \(|L||T|\), with entries ordered by location and then technology. The continuous logarithmic-spiral movement is converted to a binary position using a sigmoid threshold, after which the one-technology-per-location restriction is repaired. The fitness function assigns infeasible, over-budget configurations fitness zero.

For moth position \(M_i\), its selected flame position \(F_j\), distance \(D_i=\lVert F_j-M_i\rVert_2\), random scalar \(t\in[-1,1]\), and spiral factor \(b\), the movement used here is

$$
M_i^{\mathrm{new}}=M_i+D_i e^{bt}\cos(2\pi t)\frac{F_j-M_i}{D_i+10^{-10}}.
$$

The small denominator term keeps the computation defined when the two positions coincide. Each component is transformed by \(s(z)=1/(1+e^{-z})\) and thresholded at 0.5 to obtain a binary vector. The code implements this step in `logarithmic_spiral`.

At iteration \(r\), the number of flames is reduced according to

$$
N_f(r)=\max\left(1,\operatorname{round}\left(N-r\frac{N-1}{R}\right)\right),
$$

where \(N\) is the current number of ranked flames and \(R\) is the maximum number of iterations. Moth \(i\) follows flame \(i\bmod N_f(r)\). The implementation records the best-so-far fitness, the current population's average fitness, and its average pairwise Euclidean distance (diversity). These are diagnostics, not a proof of convergence to a global optimum.

With \(N\) moths, dimension \(d=|L||T|\), and \(R\) iterations, movement and fitness evaluation require approximately \(O(RN(d+|P||T||L|))\) work in this implementation. The pairwise diversity calculation adds \(O(RN^2d)\) work. These are per-run bounds; stochastic results can vary between runs unless the random-number generators are controlled.

In [ ]:
from dataclasses import dataclass
import math
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Moth:
    """A candidate deployment configuration in the MFO population."""

    position: np.ndarray
    fitness: float
    detection_rate: float
    total_cost: float
    false_alarm_rate: float


@dataclass
class Flame:
    """A ranked candidate configuration used to guide moth movement."""

    position: np.ndarray
    fitness: float
    detection_rate: float
    total_cost: float
    false_alarm_rate: float


class MothFlameOptimizer:
    """Moth-Flame Optimization for the security-inspection instance."""

    def __init__(
        self,
        locations: List[str],
        passenger_types: List[str],
        technologies: List[str],
        population_size: int = 30,
        max_iterations: int = 100,
        spiral_factor: float = 1.0,
    ):
        if population_size < 1:
            raise ValueError("population_size must be at least 1")
        if max_iterations < 1:
            raise ValueError("max_iterations must be at least 1")
        if not locations or not passenger_types or not technologies:
            raise ValueError("Locations, passenger types, and technologies must be non-empty")

        self.locations = locations
        self.passenger_types = passenger_types
        self.technologies = technologies
        self.population_size = population_size
        self.max_iterations = max_iterations
        self.spiral_factor = spiral_factor
        self.dimension = len(locations) * len(technologies)

        self.best_fitness_history: List[float] = []
        self.average_fitness_history: List[float] = []
        self.diversity_history: List[float] = []

        self.compatibility = {
            ("PreCheck", "Basic"): True,
            ("PreCheck", "Advanced"): True,
            ("PreCheck", "Enhanced"): False,
            ("Standard", "Basic"): True,
            ("Standard", "Advanced"): True,
            ("Standard", "Enhanced"): True,
            ("Selectee", "Basic"): False,
            ("Selectee", "Advanced"): True,
            ("Selectee", "Enhanced"): True,
        }

    def decode_position(self, position: np.ndarray) -> Dict[Tuple[str, str], bool]:
        """Map the ordered binary vector to location-technology decisions."""
        if len(position) != self.dimension:
            raise ValueError("Position length does not match the problem dimension")

        deployments = {}
        for i, location in enumerate(self.locations):
            for j, technology in enumerate(self.technologies):
                index = i * len(self.technologies) + j
                deployments[(location, technology)] = bool(position[index])
        return deployments

    def encode_position(self, deployments: Dict[Tuple[str, str], bool]) -> np.ndarray:
        """Map deployment decisions to the ordered binary-vector encoding."""
        position = np.zeros(self.dimension, dtype=float)
        for i, location in enumerate(self.locations):
            for j, technology in enumerate(self.technologies):
                index = i * len(self.technologies) + j
                position[index] = float(deployments.get((location, technology), False))
        return position

    def validate_position(self, position: np.ndarray) -> np.ndarray:
        """Repair a vector so that no location receives multiple technologies."""
        validated = np.asarray(position, dtype=float).copy()
        if len(validated) != self.dimension:
            raise ValueError("Position length does not match the problem dimension")

        for i in range(len(self.locations)):
            start = i * len(self.technologies)
            end = start + len(self.technologies)
            location_slice = validated[start:end]
            selected = np.flatnonzero(location_slice > 0.5)
            if len(selected) > 1:
                # A deterministic repair keeps the first selected technology in the defined encoding order.
                validated[start:end] = 0.0
                validated[start + selected[0]] = 1.0
        return (validated > 0.5).astype(float)

    def calculate_fitness(
        self, position: np.ndarray, problem_data: Dict
    ) -> Tuple[float, float, float, float]:
        """Return fitness, detection rate, deployment cost, and false-alarm rate."""
        position = np.asarray(position, dtype=float)
        if len(position) != self.dimension:
            raise ValueError("Position length does not match the problem dimension")
        if not np.all(np.isin(position, [0.0, 1.0])):
            raise ValueError("Fitness evaluation requires a binary position")

        deployments = self.decode_position(position)
        total_cost = sum(
            problem_data["deployment_costs"][(location, technology)]
            for (location, technology), deployed in deployments.items()
            if deployed
        )

        if total_cost > problem_data["total_budget"]:
            return 0.0, 0.0, float(total_cost), 1.0

        total_detection_benefit = 0.0
        total_false_alarms = 0.0

        for passenger_type in self.passenger_types:
            demand = problem_data["demands"][passenger_type]
            threat_probability = problem_data["threat_probabilities"][passenger_type]
            best_detection = 0.0
            best_false_alarm = 0.0

            for technology in self.technologies:
                compatible = self.compatibility.get((passenger_type, technology), False)
                deployed_anywhere = any(
                    deployments.get((location, technology), False)
                    for location in self.locations
                )
                if compatible and deployed_anywhere:
                    detection = problem_data["detection_rates"][(passenger_type, technology)]
                    if detection > best_detection:
                        best_detection = detection
                        best_false_alarm = problem_data["false_alarm_rates"][
                            (passenger_type, technology)
                        ]

            total_detection_benefit += threat_probability * best_detection * demand
            total_false_alarms += best_false_alarm * demand

        fitness = total_detection_benefit - 0.1 * total_false_alarms
        total_threats = sum(
            problem_data["threat_probabilities"][passenger_type]
            * problem_data["demands"][passenger_type]
            for passenger_type in self.passenger_types
        )
        total_passengers = sum(
            problem_data["demands"][passenger_type]
            for passenger_type in self.passenger_types
        )
        detection_rate = (
            total_detection_benefit / total_threats if total_threats > 0 else 0.0
        )
        false_alarm_rate = (
            total_false_alarms / total_passengers if total_passengers > 0 else 0.0
        )
        return fitness, detection_rate, float(total_cost), false_alarm_rate

    def initialize_population(self, problem_data: Dict) -> List[Moth]:
        """Generate and evaluate random, repaired binary candidate positions."""
        population = []
        for _ in range(self.population_size):
            binary_position = (np.random.random(self.dimension) > 0.5).astype(float)
            position = self.validate_position(binary_position)
            fitness, detection, cost, false_alarm = self.calculate_fitness(
                position, problem_data
            )
            population.append(
                Moth(position, fitness, detection, cost, false_alarm)
            )
        return population

    def logarithmic_spiral(
        self,
        moth_pos: np.ndarray,
        flame_pos: np.ndarray,
        iteration: int,
        max_iterations: int,
    ) -> np.ndarray:
        """Move a moth relative to its selected flame, then binarize and repair."""
        del iteration, max_iterations  # The spiral rule used here does not vary with iteration.
        distance = np.linalg.norm(flame_pos - moth_pos)
        t = np.random.uniform(-1.0, 1.0)
        spiral_movement = distance * np.exp(self.spiral_factor * t) * np.cos(2.0 * np.pi * t)
        new_position = moth_pos + spiral_movement * (flame_pos - moth_pos) / (distance + 1e-10)

        # The sigmoid maps continuous coordinates to probabilities before thresholding.
        sigmoid = 1.0 / (1.0 + np.exp(-new_position))
        binary_position = (sigmoid > 0.5).astype(float)
        return self.validate_position(binary_position)

    def update_flames(self, moths: List[Moth]) -> List[Flame]:
        """Rank the current moths and copy their positions into flame records."""
        sorted_moths = sorted(moths, key=lambda moth: moth.fitness, reverse=True)
        return [
            Flame(
                moth.position.copy(),
                moth.fitness,
                moth.detection_rate,
                moth.total_cost,
                moth.false_alarm_rate,
            )
            for moth in sorted_moths
        ]

    def calculate_diversity(self, population: List[Moth]) -> float:
        """Measure the mean pairwise Euclidean distance in the current population."""
        if len(population) <= 1:
            return 0.0

        total_distance = 0.0
        count = 0
        for i in range(len(population)):
            for j in range(i + 1, len(population)):
                total_distance += np.linalg.norm(
                    population[i].position - population[j].position
                )
                count += 1
        return total_distance / count

    def optimize(
        self, problem_data: Dict
    ) -> Tuple[Flame, List[float], List[float], List[float]]:
        """Run MFO and return the best recorded flame and search histories."""
        print("Starting Moth-Flame Optimization")
        print(f"Population size: {self.population_size}")
        print(f"Maximum iterations: {self.max_iterations}")
        print(f"Problem dimension: {self.dimension}")

        population = self.initialize_population(problem_data)
        self.best_fitness_history = []
        self.average_fitness_history = []
        self.diversity_history = []
        best_flame_ever: Optional[Flame] = None

        for iteration in range(self.max_iterations):
            flames = self.update_flames(population)
            if flames and (
                best_flame_ever is None
                or flames[0].fitness > best_flame_ever.fitness
            ):
                best_flame_ever = Flame(
                    flames[0].position.copy(),
                    flames[0].fitness,
                    flames[0].detection_rate,
                    flames[0].total_cost,
                    flames[0].false_alarm_rate,
                )

            best_fitness = max(moth.fitness for moth in population)
            average_fitness = sum(moth.fitness for moth in population) / len(population)
            diversity = self.calculate_diversity(population)
            self.best_fitness_history.append(best_flame_ever.fitness)
            self.average_fitness_history.append(average_fitness)
            self.diversity_history.append(diversity)

            # Reducing the flame count shifts guidance from a broad set of candidates to top-ranked flames.
            flame_number = max(
                1,
                round(
                    len(flames)
                    - (iteration + 1) * (len(flames) - 1) / self.max_iterations
                ),
            )
            new_population = []
            for i, moth in enumerate(population):
                selected_flame = flames[i % flame_number]
                new_position = self.logarithmic_spiral(
                    moth.position,
                    selected_flame.position,
                    iteration,
                    self.max_iterations,
                )
                fitness, detection, cost, false_alarm = self.calculate_fitness(
                    new_position, problem_data
                )
                new_population.append(
                    Moth(new_position, fitness, detection, cost, false_alarm)
                )
            population = new_population

            if (iteration + 1) % 10 == 0 or iteration == 0:
                print(
                    f"Iteration {iteration + 1:3d}: "
                    f"Best-so-far fitness = {best_flame_ever.fitness:.6f}, "
                    f"Average fitness = {average_fitness:.6f}, "
                    f"Diversity = {diversity:.4f}"
                )

        if best_flame_ever is None:
            raise RuntimeError("The optimizer did not evaluate any candidate")

        print("\nMoth-Flame Optimization completed")
        print(f"Best fitness: {best_flame_ever.fitness:.6f}")
        print(f"Detection rate: {best_flame_ever.detection_rate * 100:.2f}%")
        print(f"Total cost: ${best_flame_ever.total_cost:,.0f}")
        print(f"False-alarm rate: {best_flame_ever.false_alarm_rate * 100:.2f}%")
        return (
            best_flame_ever,
            self.best_fitness_history,
            self.average_fitness_history,
            self.diversity_history,
        )

In [ ]:
def create_mfo_problem_instance() -> Dict:
    """Create the specified security-inspection example instance."""
    locations = [
        "Location_1",
        "Location_2",
        "Location_3",
        "Location_4",
        "Location_5",
    ]
    passenger_types = ["PreCheck", "Standard", "Selectee"]
    technologies = ["Basic", "Advanced", "Enhanced"]

    demands = {"PreCheck": 5000, "Standard": 15000, "Selectee": 1000}
    threat_probabilities = {
        "PreCheck": 0.0001,
        "Standard": 0.001,
        "Selectee": 0.01,
    }
    detection_rates = {
        ("PreCheck", "Basic"): 0.85,
        ("PreCheck", "Advanced"): 0.92,
        ("PreCheck", "Enhanced"): 0.98,
        ("Standard", "Basic"): 0.75,
        ("Standard", "Advanced"): 0.88,
        ("Standard", "Enhanced"): 0.96,
        ("Selectee", "Basic"): 0.65,
        ("Selectee", "Advanced"): 0.82,
        ("Selectee", "Enhanced"): 0.94,
    }
    false_alarm_rates = {
        ("PreCheck", "Basic"): 0.05,
        ("PreCheck", "Advanced"): 0.08,
        ("PreCheck", "Enhanced"): 0.12,
        ("Standard", "Basic"): 0.08,
        ("Standard", "Advanced"): 0.12,
        ("Standard", "Enhanced"): 0.18,
        ("Selectee", "Basic"): 0.15,
        ("Selectee", "Advanced"): 0.20,
        ("Selectee", "Enhanced"): 0.25,
    }
    deployment_costs = {
        ("Location_1", "Basic"): 50000,
        ("Location_1", "Advanced"): 75000,
        ("Location_1", "Enhanced"): 120000,
        ("Location_2", "Basic"): 60000,
        ("Location_2", "Advanced"): 100000,
        ("Location_2", "Enhanced"): 150000,
        ("Location_3", "Basic"): 55000,
        ("Location_3", "Advanced"): 90000,
        ("Location_3", "Enhanced"): 200000,
        ("Location_4", "Basic"): 65000,
        ("Location_4", "Advanced"): 110000,
        ("Location_4", "Enhanced"): 180000,
        ("Location_5", "Basic"): 70000,
        ("Location_5", "Advanced"): 120000,
        ("Location_5", "Enhanced"): 220000,
    }
    return {
        "locations": locations,
        "passenger_types": passenger_types,
        "technologies": technologies,
        "demands": demands,
        "threat_probabilities": threat_probabilities,
        "detection_rates": detection_rates,
        "false_alarm_rates": false_alarm_rates,
        "deployment_costs": deployment_costs,
        "total_budget": 500000,
    }


mfo_problem_data = create_mfo_problem_instance()
print("MFO problem instance created:")
print(f"  Locations: {len(mfo_problem_data['locations'])}")
print(f"  Passenger types: {len(mfo_problem_data['passenger_types'])}")
print(f"  Technologies: {len(mfo_problem_data['technologies'])}")
print(f"  Total budget: ${mfo_problem_data['total_budget']:,}")

In [ ]:
mfo = MothFlameOptimizer(
    locations=mfo_problem_data["locations"],
    passenger_types=mfo_problem_data["passenger_types"],
    technologies=mfo_problem_data["technologies"],
    population_size=30,
    max_iterations=100,
    spiral_factor=1.0,
)

start_time = time.time()
best_flame, fitness_history, avg_fitness_history, diversity_history = mfo.optimize(
    mfo_problem_data
)
execution_time = time.time() - start_time
print(f"\nExecution time: {execution_time:.2f} seconds")

best_deployments = mfo.decode_position(best_flame.position)
selected_deployments = [
    (location, technology)
    for (location, technology), deployed in best_deployments.items()
    if deployed
]

# These checks link the reported candidate to the model's feasibility rules and objective calculation.
assert len(selected_deployments) <= len(mfo_problem_data["locations"])
for location in mfo_problem_data["locations"]:
    assert sum(
        best_deployments[(location, technology)]
        for technology in mfo_problem_data["technologies"]
    ) <= 1
assert best_flame.total_cost <= mfo_problem_data["total_budget"]
recalculated = mfo.calculate_fitness(best_flame.position, mfo_problem_data)
assert math.isclose(recalculated[0], best_flame.fitness, rel_tol=1e-12, abs_tol=1e-12)
assert math.isclose(recalculated[2], best_flame.total_cost, rel_tol=0.0, abs_tol=1e-9)
assert math.isclose(recalculated[1], best_flame.detection_rate, rel_tol=1e-12, abs_tol=1e-12)
assert math.isclose(recalculated[3], best_flame.false_alarm_rate, rel_tol=1e-12, abs_tol=1e-12)

print("\n=== BEST RECORDED CONFIGURATION ===")
print(f"Fitness: {best_flame.fitness:.6f}")
print(f"Detection rate: {best_flame.detection_rate * 100:.2f}%")
print(f"False-alarm rate: {best_flame.false_alarm_rate * 100:.2f}%")
print(f"Total cost: ${best_flame.total_cost:,.0f}")
print(
    f"Budget utilization: "
    f"{best_flame.total_cost / mfo_problem_data['total_budget'] * 100:.1f}%"
)
print("\nDeployment configuration:")
for location, technology in selected_deployments:
    cost = mfo_problem_data["deployment_costs"][(location, technology)]
    print(f"  {location}: {technology} (cost: ${cost:,})")
print(f"\nTotal deployments: {len(selected_deployments)}")

In [ ]:
def create_mfo_visualizations(
    best_flame: Flame,
    fitness_history: List[float],
    avg_fitness_history: List[float],
    diversity_history: List[float],
    mfo_problem_data: Dict,
    mfo: MothFlameOptimizer,
) -> None:
    """Plot the best candidate, search histories, and deployment encoding."""
    if not fitness_history:
        raise ValueError("Search histories are empty; run the optimizer first")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Moth-Flame Optimization Results", fontsize=16, fontweight="bold")
    iterations = range(1, len(fitness_history) + 1)

    ax = axes[0, 0]
    ax.plot(iterations, fitness_history, label="Best-so-far fitness", linewidth=2)
    ax.plot(iterations, avg_fitness_history, "--", label="Current-population average", linewidth=2)
    ax.set_title("Fitness History")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Fitness")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(iterations, diversity_history, color="purple", linewidth=2)
    ax.set_title("Population Diversity")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Mean pairwise Euclidean distance")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    metric_names = ["Detection rate", "False-alarm rate"]
    metric_values = [best_flame.detection_rate, best_flame.false_alarm_rate]
    bars = ax.bar(metric_names, [value * 100 for value in metric_values], color=["green", "red"])
    ax.set_title("Best Candidate Performance")
    ax.set_ylabel("Rate (%)")
    ax.set_ylim(0, max(1.0, max(metric_values) * 100 * 1.2))
    for bar, value in zip(bars, metric_values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value * 100:.2f}%",
            ha="center",
            va="bottom",
        )
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[1, 1]
    budget_used = best_flame.total_cost
    budget_remaining = mfo_problem_data["total_budget"] - budget_used
    ax.pie(
        [budget_used, budget_remaining],
        labels=["Budget used", "Budget remaining"],
        autopct="%1.1f%%",
        startangle=90,
    )
    ax.set_title(f"Budget Allocation (total: ${mfo_problem_data['total_budget']:,})")
    plt.tight_layout()
    plt.show()

    deployment_matrix = np.zeros(
        (len(mfo_problem_data["locations"]), len(mfo_problem_data["technologies"]))
    )
    deployments = mfo.decode_position(best_flame.position)
    for (location, technology), deployed in deployments.items():
        if deployed:
            row = mfo_problem_data["locations"].index(location)
            column = mfo_problem_data["technologies"].index(technology)
            deployment_matrix[row, column] = 1

    fig, ax = plt.subplots(figsize=(12, 6))
    image = ax.imshow(deployment_matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    for row in range(deployment_matrix.shape[0]):
        for column in range(deployment_matrix.shape[1]):
            label = "Deployed" if deployment_matrix[row, column] == 1 else "Not deployed"
            ax.text(column, row, label, ha="center", va="center", color="black")
    ax.set_xticks(range(len(mfo_problem_data["technologies"])))
    ax.set_xticklabels(mfo_problem_data["technologies"])
    ax.set_yticks(range(len(mfo_problem_data["locations"])))
    ax.set_yticklabels(mfo_problem_data["locations"])
    ax.set_title("Best Recorded Deployment Configuration")
    ax.set_xlabel("Screening technology")
    ax.set_ylabel("Location")
    colorbar = plt.colorbar(image, ax=ax, ticks=[0, 1])
    colorbar.set_label("Deployment status (0 = not deployed, 1 = deployed)")
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Search Diagnostics", fontsize=16, fontweight="bold")

    improvement = np.diff(fitness_history)
    axes[0].plot(range(2, len(fitness_history) + 1), improvement, linewidth=2)
    axes[0].axhline(0, color="black", linestyle="--", alpha=0.7)
    axes[0].set_title("Best-so-far Fitness Change")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Change in fitness")
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(diversity_history, fitness_history, alpha=0.7)
    axes[1].set_title("Diversity and Best-so-far Fitness")
    axes[1].set_xlabel("Mean pairwise distance")
    axes[1].set_ylabel("Best-so-far fitness")
    axes[1].grid(True, alpha=0.3)

    final_fitness = fitness_history[-1]
    if final_fitness > 0:
        percentage = [value / final_fitness * 100 for value in fitness_history]
        axes[2].plot(iterations, percentage, linewidth=2, label="Share of final best fitness")
        axes[2].axhline(90, color="orange", linestyle="--", label="90% reference")
        axes[2].axhline(95, color="red", linestyle="--", label="95% reference")
        axes[2].legend()
    axes[2].set_title("Best-so-far Fitness Progress")
    axes[2].set_xlabel("Iteration")
    axes[2].set_ylabel("Fitness as a percentage of final value (%)")
    axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


create_mfo_visualizations(
    best_flame,
    fitness_history,
    avg_fitness_history,
    diversity_history,
    mfo_problem_data,
    mfo,
)

In [ ]:
def summarize_mfo_result(best_flame: Flame, problem_data: Dict) -> pd.DataFrame:
    """Present the MFO result and verify its reported feasibility metrics."""
    summary = pd.DataFrame(
        [
            {
                "Method": "Moth-Flame Optimization",
                "Best-so-far fitness": best_flame.fitness,
                "Detection rate (%)": best_flame.detection_rate * 100,
                "False-alarm rate (%)": best_flame.false_alarm_rate * 100,
                "Deployment cost ($)": best_flame.total_cost,
                "Budget utilization (%)": (
                    best_flame.total_cost / problem_data["total_budget"] * 100
                ),
                "Execution time (s)": execution_time,
            }
        ]
    )
    print("=== MFO RESULT SUMMARY ===")
    print(summary.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    print(
        "\nInterpretation: the fitness is the threat-weighted detection benefit minus "
        "the specified false-alarm penalty. Cost is checked as a separate hard constraint."
    )
    print(
        "The recorded candidate is the best encountered during this stochastic run; "
        "the result is not an optimality certificate."
    )
    return summary


mfo_result_summary = summarize_mfo_result(best_flame, mfo_problem_data)

In [ ]:
def parameter_sensitivity_analysis(problem_data: Dict) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Examine how selected MFO settings affect one stochastic run per setting."""
    print("=== PARAMETER SENSITIVITY ANALYSIS ===")

    population_sizes = [10, 20, 30, 40, 50]
    population_results = []
    print("Testing population sizes...")
    for population_size in population_sizes:
        optimizer = MothFlameOptimizer(
            locations=problem_data["locations"],
            passenger_types=problem_data["passenger_types"],
            technologies=problem_data["technologies"],
            population_size=population_size,
            max_iterations=50,
            spiral_factor=1.0,
        )
        start = time.time()
        best, _, _, _ = optimizer.optimize(problem_data)
        elapsed = time.time() - start
        population_results.append(
            {
                "Population size": population_size,
                "Detection rate (%)": best.detection_rate * 100,
                "Fitness": best.fitness,
                "Time (s)": elapsed,
            }
        )
        print(
            f"  Population {population_size}: detection "
            f"{best.detection_rate * 100:.2f}%, elapsed {elapsed:.2f} s"
        )

    spiral_factors = [0.5, 0.8, 1.0, 1.2, 1.5]
    spiral_results = []
    print("\nTesting spiral factors...")
    for spiral_factor in spiral_factors:
        optimizer = MothFlameOptimizer(
            locations=problem_data["locations"],
            passenger_types=problem_data["passenger_types"],
            technologies=problem_data["technologies"],
            population_size=30,
            max_iterations=50,
            spiral_factor=spiral_factor,
        )
        start = time.time()
        best, _, _, _ = optimizer.optimize(problem_data)
        elapsed = time.time() - start
        spiral_results.append(
            {
                "Spiral factor": spiral_factor,
                "Detection rate (%)": best.detection_rate * 100,
                "Fitness": best.fitness,
                "Time (s)": elapsed,
            }
        )
        print(
            f"  Spiral factor {spiral_factor}: detection "
            f"{best.detection_rate * 100:.2f}%, elapsed {elapsed:.2f} s"
        )

    population_frame = pd.DataFrame(population_results)
    spiral_frame = pd.DataFrame(spiral_results)
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("MFO Parameter Sensitivity: Single-Run Observations", fontsize=15)

    axes[0, 0].plot(
        population_frame["Population size"],
        population_frame["Detection rate (%)"],
        "o-",
    )
    axes[0, 0].set_title("Population Size and Detection Rate")
    axes[0, 0].set_xlabel("Population size (moths)")
    axes[0, 0].set_ylabel("Detection rate (%)")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(population_frame["Population size"], population_frame["Time (s)"], "o-")
    axes[0, 1].set_title("Population Size and Runtime")
    axes[0, 1].set_xlabel("Population size (moths)")
    axes[0, 1].set_ylabel("Runtime (seconds)")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(
        spiral_frame["Spiral factor"], spiral_frame["Detection rate (%)"], "o-"
    )
    axes[1, 0].set_title("Spiral Factor and Detection Rate")
    axes[1, 0].set_xlabel("Spiral factor")
    axes[1, 0].set_ylabel("Detection rate (%)")
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(spiral_frame["Spiral factor"], spiral_frame["Time (s)"], "o-")
    axes[1, 1].set_title("Spiral Factor and Runtime")
    axes[1, 1].set_xlabel("Spiral factor")
    axes[1, 1].set_ylabel("Runtime (seconds)")
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(
        "Each plotted point is one stochastic run. Differences can reflect both "
        "the parameter setting and random variation; the curves are not statistical estimates."
    )
    return population_frame, spiral_frame


pop_sensitivity, spiral_sensitivity = parameter_sensitivity_analysis(mfo_problem_data)

## Interpreting the method

MFO represents each candidate deployment as a moth and uses ranked candidate configurations as flames. The logarithmic-spiral rule provides the movement mechanism, while the decreasing flame count changes how many high-ranking candidates guide the population. Binary thresholding and position repair connect this continuous movement rule to the discrete deployment decisions.

### Strengths and limitations

- A population of candidates explores several configurations during a run.
- The method can be applied when the objective and feasibility checks are available but a direct exact solution procedure is not being used in this tier.
- The recorded best-so-far solution is retained for reporting even if later population updates produce lower-fitness candidates.
- The algorithm does not guarantee a globally optimal solution. Its output depends on stochastic initialization and movement, and repeated runs can differ.
- Runtime grows with population size, iteration count, problem dimension, and the pairwise-diversity calculation.
- Parameter sensitivity observations from one run per setting are descriptive only; they do not establish that one setting is generally superior.

### Reading the figures and result

The fitness-history plot distinguishes the best-so-far candidate from the average fitness of the current population. The diversity plot measures separation among current binary position vectors; low diversity indicates that candidates have become similar, not that the global optimum has been found. The deployment heatmap displays the selected binary decisions, and the budget chart shows how much of the budget the recorded feasible candidate uses. Detection rate, false-alarm rate, cost, and fitness describe different aspects of the model and should be read together.

### Summary

This tier demonstrates a Moth-Flame Optimization search for the stated binary screening configuration problem. Its result is a best-found feasible candidate under the instance's objective, compatibility rules, and budget—not an optimality certificate. The analysis is deliberately limited to the aggregate availability and rate assumptions specified above. Additional real-world features, such as location-specific passenger flows, capacity constraints, or further operational requirements, would need to be defined as explicit model inputs and constraints before their effects could be assessed.